In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
sys.path.append('/content/drive/MyDrive')

In [3]:
import os
print(os.listdir('/content/drive/MyDrive'))

['ML awfera course', 'Colab Notebooks', 'Copy of ExploratoryDataAnalysis(EDA).ipynb', '.ipynb_checkpoints', 'Web demo recording ', 'Machine Learning Track', 'Internship Questionnaire.docx', 'Computer vision', 'Aariz', 'landmark_model.pth', 'config.py', 'dataset.py']


In [4]:
import cv2
import torch
import torch.nn as nn
# ... other imports

In [5]:
import os
print(os.listdir())

['.config', 'config.py', 'dataset.py', 'drive', 'sample_data']


In [6]:
from config import *
from dataset import AarizDataset

In [7]:
print(NUM_LANDMARKS)

29


In [8]:
BASE_PATH = "/content/drive/MyDrive/Aariz"

train_dataset = AarizDataset(BASE_PATH, "TRAIN")
val_dataset   = AarizDataset(BASE_PATH, "VALID")
test_dataset  = AarizDataset(BASE_PATH, "TEST")

In [9]:
print("Train size:", len(train_dataset))

img, landmarks, cvm = train_dataset[0]

print("Image shape:", img.shape)        # (H, W, 3)
print("Landmarks shape:", landmarks.shape)  # (1, 29, 2)
print("CVM shape:", cvm.shape)          # (1, 6)

Train size: 700
Image shape: (2225, 1968, 3)
Landmarks shape: (1, 29, 2)
CVM shape: (1, 6)


In [10]:
import torch
from torch.utils.data import Dataset

class TorchDataset(Dataset):
    def __init__(self, aariz_dataset):
        self.dataset = aariz_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, landmarks, cvm = self.dataset[idx]

        # Resize + normalize
        image = cv2.resize(image, (256, 256))
        image = image / 255.0

        # Convert to tensor
        image = torch.tensor(image).permute(2,0,1).float()  # (3, H, W)

        # Normalize landmarks
        h, w = 256, 256
        landmarks = landmarks[0]
        landmarks[:, 0] /= w
        landmarks[:, 1] /= h

        landmarks = torch.tensor(landmarks.flatten()).float()

        return image, landmarks

In [14]:
from torch.utils.data import DataLoader

train_loader = DataLoader(TorchDataset(train_dataset), batch_size=8, shuffle=True)
val_loader   = DataLoader(TorchDataset(val_dataset), batch_size=8)

In [15]:
import torchvision.models as models
import torch.nn as nn

class LandmarkModel(nn.Module):
    def __init__(self, num_landmarks):
        super().__init__()
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.fc = nn.Linear(512, num_landmarks * 2)

    def forward(self, x):
        return self.backbone(x)

In [16]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

num_landmarks = NUM_LANDMARKS

model = LandmarkModel(num_landmarks).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

for epoch in range(20):
    model.train()
    total_loss = 0

    for imgs, targets in train_loader:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)

        preds = model(imgs)
        loss = criterion(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 6.7525
Epoch 2, Loss: 0.4785
Epoch 3, Loss: 0.2166
Epoch 4, Loss: 0.1277
Epoch 5, Loss: 0.1208
Epoch 6, Loss: 0.0864
Epoch 7, Loss: 0.0661
Epoch 8, Loss: 0.0647
Epoch 9, Loss: 0.0591
Epoch 10, Loss: 0.0419
Epoch 11, Loss: 0.0421
Epoch 12, Loss: 0.0392
Epoch 13, Loss: 0.0446
Epoch 14, Loss: 0.0390
Epoch 15, Loss: 0.0397
Epoch 16, Loss: 0.0271
Epoch 17, Loss: 0.0247
Epoch 18, Loss: 0.0441
Epoch 19, Loss: 0.0378
Epoch 20, Loss: 0.0258


In [45]:
print(preds.shape)

torch.Size([1, 58])


In [17]:
import numpy as np

def preds_to_landmarks(preds):
    preds = preds.cpu().detach().numpy()
    preds = preds.reshape(-1, 29, 2)   # (batch, 29, 2)
    return preds

In [18]:
# create index → symbol mapping
landmark_symbols = [v["symbol"] for v in ANATOMICAL_LANDMARKS.values()]

In [19]:
def landmarks_to_dict(landmarks):
    lm_dict = {}
    for i, (x, y) in enumerate(landmarks):
        lm_dict[landmark_symbols[i]] = (x, y)
    return lm_dict

In [20]:
import numpy as np

def calculate_angle(A, B, C):
    A, B, C = np.array(A), np.array(B), np.array(C)

    BA = A - B
    BC = C - B

    cos_angle = np.dot(BA, BC) / (np.linalg.norm(BA) * np.linalg.norm(BC))
    angle = np.degrees(np.arccos(cos_angle))

    return angle

In [21]:
def compute_angles(lm):
    S = lm["S"]
    N = lm["N"]
    A = lm["A"]
    B = lm["B"]

    SNA = calculate_angle(S, N, A)
    SNB = calculate_angle(S, N, B)
    ANB = SNA - SNB

    return SNA, SNB, ANB

In [22]:
def generate_report(SNA, SNB, ANB):
    report = []

    report.append(f"SNA: {SNA:.2f}")
    report.append(f"SNB: {SNB:.2f}")
    report.append(f"ANB: {ANB:.2f}")

    # Diagnosis
    if ANB > 4:
        diagnosis = "Class II (Retruded mandible)"
    elif ANB < 2:
        diagnosis = "Class III (Prognathic mandible)"
    else:
        diagnosis = "Class I (Normal)"

    report.append(f"Diagnosis: {diagnosis}")

    return "\n".join(report)

In [23]:
def predict_and_report(model, dataset, idx=0):
    model.eval()

    image, _, _ = dataset[idx]

    # preprocess
    img = cv2.resize(image, (256, 256)) / 255.0
    img = torch.tensor(img).permute(2,0,1).unsqueeze(0).float().to(DEVICE)

    with torch.no_grad():
        preds = model(img)

    landmarks = preds_to_landmarks(preds)[0]

    lm_dict = landmarks_to_dict(landmarks)

    SNA, SNB, ANB = compute_angles(lm_dict)

    report = generate_report(SNA, SNB, ANB)

    print("\n📋 ORTHODONTIC REPORT")
    print("---------------------")
    print(report)

    return lm_dict

In [46]:
lm = predict_and_report(model, train_dataset, idx=9)


📋 ORTHODONTIC REPORT
---------------------
SNA: 81.49
SNB: 74.07
ANB: 7.42
Diagnosis: Class II (Retruded mandible)


In [26]:
import matplotlib.pyplot as plt

def visualize(image, lm_dict):
    plt.imshow(image)
    for k, (x, y) in lm_dict.items():
        plt.scatter(x, y, c='red')
        plt.text(x, y, k, color='yellow')
    plt.show()

In [47]:
from google.colab import files
uploaded = files.upload()

Saving cks2ip8fr2a2r0yuf718nerfe.png to cks2ip8fr2a2r0yuf718nerfe.png


In [55]:
torch.save(model.state_dict(), "model.pth")

In [56]:
from google.colab import files
files.download("model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:
import torch

model.load_state_dict(torch.load("model.pth", map_location="cpu"))
model.eval()

LandmarkModel(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, tr

In [62]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image = cv2.imread(file_name)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# ✅ define here
orig_h, orig_w = image.shape[:2]

In [63]:
img = cv2.resize(image, (256, 256))
img = img / 255.0
img = np.transpose(img, (2,0,1))
img = np.expand_dims(img, axis=0)

In [64]:
with torch.no_grad():
    preds = model(torch.tensor(img).float())

landmarks = preds.squeeze().numpy().reshape(-1, 2)

In [65]:
landmarks[:, 0] *= orig_w
landmarks[:, 1] *= orig_h

In [66]:
import numpy as np

def angle(p1, p2, p3):
    a = np.array(p1)
    b = np.array(p2)
    c = np.array(p3)

    ba = a - b
    bc = c - b

    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba)*np.linalg.norm(bc))
    return np.degrees(np.arccos(cos_angle))

In [67]:
def report(SNA, SNB, ANB):
    print("===== REPORT =====")
    print(f"SNA: {SNA:.2f}")
    print(f"SNB: {SNB:.2f}")
    print(f"ANB: {ANB:.2f}")

    if ANB > 4:
        print("Class II Skeletal")
    elif ANB < 0:
        print("Class III Skeletal")
    else:
        print("Class I Skeletal")

In [68]:
report(SNA, SNB, ANB)

===== REPORT =====
SNA: 75.86
SNB: 72.18
ANB: 3.68
Class I Skeletal
